# Task3: Байесовские модели для прогнозирования теннисных матчей

## Цель
- Выбрать PS (Pinnacle) как baseline с покрытием 97% и низкой маржой ~2.5%
- Найти слабые места букмекера (ошибки по поверхности, рангу, фаворит/андердог)
- Протестировать байесовские модели: GaussianNB, CategoricalNB, калибровка
- Попытаться превзойти baseline

## Данные
- `ods.parquet`: odds букмекеров
- `tml.parquet`: матчи ATP
- `players.parquet`: игроки

## План
1. Джойн данных по дате + именам
2. Признаки: implied prob PS, diff rank, surface, h2h, form
3. Split: train <2024, test >=2024
4. Baseline: PS implied prob
5. Модели: GNB (с/без scale), CategoricalNB для surface, калибровка

In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB, CategoricalNB
from sklearn.metrics import log_loss, roc_auc_score, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV
import matplotlib.pyplot as plt

# Load data
ods = pd.read_parquet('ods.parquet')
tml = pd.read_parquet('tml.parquet')
players = pd.read_parquet('players.parquet')

print('ods shape:', ods.shape)
print('tml shape:', tml.shape)
print('players shape:', players.shape)

# Normalize dates
ods['Date'] = pd.to_datetime(ods['Date'])
tml['tourney_date'] = pd.to_datetime(tml['tourney_date'])

# Rename for consistency
ods = ods.rename(columns={'Date': 'date', 'Winner': 'winner_name', 'Loser': 'loser_name'})
tml = tml.rename(columns={'tourney_date': 'date'})

# Left join on date + names (as in previous analyses)
merged = pd.merge(tml, ods, on=['date', 'winner_name', 'loser_name'], how='left')
print('merged shape:', merged.shape)
print('merged has PS odds:', merged['PSW'].notna().sum())

# Filter to PS available
ps_data = merged.dropna(subset=['PSW', 'PSL']).copy()
print('ps_data shape:', ps_data.shape)

ods shape: (22851, 42)
tml shape: (24939, 50)
players shape: (7643, 12)
merged shape: (24939, 89)
merged has PS odds: 0
ps_data shape: (0, 89)


In [10]:
# Features
ps_data['p_ps_w'] = 1 / ps_data['PSW']
ps_data['p_ps_l'] = 1 / ps_data['PSL']
ps_data['overround'] = ps_data['p_ps_w'] + ps_data['p_ps_l']
ps_data['p_ps_imp'] = ps_data['p_ps_w'] / ps_data['overround']  # implied prob for winner (P1)

# Target: P1 wins
ps_data['y_p1_win'] = 1

# Features
ps_data['rank_diff_p1_minus_p2'] = ps_data['winner_rank'] - ps_data['loser_rank']  # negative if winner higher
ps_data['rank_points_diff_p1_minus_p2'] = ps_data['winner_rank_points'] - ps_data['loser_rank_points']

# Surface
ps_data['surface'] = ps_data['surface'].str.lower()

# Check dates
print('ps_data years:', sorted(ps_data['date'].dt.year.unique()))
print('ps_data date range:', ps_data['date'].min(), 'to', ps_data['date'].max())

# Split: train <2023, test >=2023 (adjust if needed)
train = ps_data[ps_data['date'] < '2023-01-01']
test = ps_data[ps_data['date'] >= '2023-01-01']

print('train shape:', train.shape)
print('test shape:', test.shape)
print('train years:', train['date'].dt.year.min(), '-', train['date'].dt.year.max())
print('test years:', test['date'].dt.year.min(), '-', test['date'].dt.year.max())

ps_data years: []
ps_data date range: NaT to NaT
train shape: (0, 96)
test shape: (0, 96)
train years: nan - nan
test years: nan - nan


In [11]:
# Baseline: PS implied prob
def compute_baseline_metrics(df, prob_col):
    y_true = df['y_p1_win']
    y_prob = df[prob_col]
    ll = log_loss(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)
    acc = accuracy_score(y_true, (y_prob > 0.5).astype(int))
    return {'log_loss': ll, 'auc': auc, 'acc': acc}

train_baseline = compute_baseline_metrics(train, 'p_ps_imp')
test_baseline = compute_baseline_metrics(test, 'p_ps_imp')

print('Train baseline (PS):', train_baseline)
print('Test baseline (PS):', test_baseline)

# Features for models
features_continuous = ['p_ps_imp', 'rank_diff_p1_minus_p2', 'rank_points_diff_p1_minus_p2']
features_cat = ['surface']

X_train_cont = train[features_continuous].fillna(0)  # simple fill
X_test_cont = test[features_continuous].fillna(0)

# Encode surface
surface_map = {'hard': 0, 'clay': 1, 'grass': 2}
train['surface_enc'] = train['surface'].map(surface_map).fillna(3)  # unknown
test['surface_enc'] = test['surface'].map(surface_map).fillna(3)

y_train = train['y_p1_win']
y_test = test['y_p1_win']

ValueError: Found array with 0 sample(s) (shape=(0,)) while a minimum of 1 is required.

In [ ]:
# Models
models = {}

# GaussianNB raw
gnb_raw = GaussianNB()
gnb_raw.fit(X_train_cont, y_train)
y_prob_gnb_raw = gnb_raw.predict_proba(X_test_cont)[:, 1]
models['GNB_raw'] = compute_baseline_metrics(test.assign(y_p1_win=y_test, p_ps_imp=y_prob_gnb_raw), 'p_ps_imp')

# GaussianNB scaled
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_cont)
X_test_scaled = scaler.transform(X_test_cont)
gnb_scaled = GaussianNB()
gnb_scaled.fit(X_train_scaled, y_train)
y_prob_gnb_scaled = gnb_scaled.predict_proba(X_test_scaled)[:, 1]
models['GNB_scaled'] = compute_baseline_metrics(test.assign(y_p1_win=y_test, p_ps_imp=y_prob_gnb_scaled), 'p_ps_imp')

# CategoricalNB for surface
cnb = CategoricalNB()
X_train_cat = train[['surface_enc']]
X_test_cat = test[['surface_enc']]
cnb.fit(X_train_cat, y_train)
y_prob_cnb = cnb.predict_proba(X_test_cat)[:, 1]
models['CNB_surface'] = compute_baseline_metrics(test.assign(y_p1_win=y_test, p_ps_imp=y_prob_cnb), 'p_ps_imp')

# Calibrated GNB (Platt)
gnb_cal = CalibratedClassifierCV(GaussianNB(), method='sigmoid')
gnb_cal.fit(X_train_cont, y_train)
y_prob_gnb_cal = gnb_cal.predict_proba(X_test_cont)[:, 1]
models['GNB_calibrated'] = compute_baseline_metrics(test.assign(y_p1_win=y_test, p_ps_imp=y_prob_gnb_cal), 'p_ps_imp')

print('Models results:')
for name, metrics in models.items():
    print(f'{name}: LL={metrics["log_loss"]:.4f}, AUC={metrics["auc"]:.4f}, ACC={metrics["acc"]:.4f}')

print('\\nBaseline PS:')
print(f'LL={test_baseline["log_loss"]:.4f}, AUC={test_baseline["auc"]:.4f}, ACC={test_baseline["acc"]:.4f}')

In [ ]:
# Analyze baseline errors: where PS makes mistakes
test['baseline_correct'] = ((test['p_ps_imp'] > 0.5) & (test['y_p1_win'] == 1)) | ((test['p_ps_imp'] < 0.5) & (test['y_p1_win'] == 0))
baseline_errors = test[~test['baseline_correct']]

print('Baseline errors:', len(baseline_errors), '/', len(test), f'({len(baseline_errors)/len(test)*100:.1f}%)')

# By surface
surface_errors = baseline_errors['surface'].value_counts()
surface_total = test['surface'].value_counts()
print('\\nErrors by surface:')
for surf in ['hard', 'clay', 'grass']:
    if surf in surface_total:
        err = surface_errors.get(surf, 0)
        total = surface_total[surf]
        pct = err / total * 100
        print(f'{surf}: {err}/{total} ({pct:.1f}%)')

# By rank diff bins
test['rank_bin'] = pd.cut(test['rank_diff_p1_minus_p2'], bins=[-1000, -100, -50, -10, 10, 50, 100, 1000], labels=['fav-100+', 'fav-100-50', 'fav-50-10', 'close', 'dog+10-50', 'dog+50-100', 'dog+100'])
rank_errors = baseline_errors['rank_bin'].value_counts()
rank_total = test['rank_bin'].value_counts()
print('\\nErrors by rank diff:')
for bin_name in rank_total.index:
    err = rank_errors.get(bin_name, 0)
    total = rank_total[bin_name]
    pct = err / total * 100
    print(f'{bin_name}: {err}/{total} ({pct:.1f}%)')

# By probability bins (fav vs underdog)
test['prob_bin'] = pd.cut(test['p_ps_imp'], bins=[0, 0.3, 0.4, 0.6, 0.7, 1], labels=['strong_dog', 'dog', 'even', 'fav', 'strong_fav'])
prob_errors = baseline_errors['prob_bin'].value_counts()
prob_total = test['prob_bin'].value_counts()
print('\\nErrors by prob bin:')
for bin_name in prob_total.index:
    err = prob_errors.get(bin_name, 0)
    total = prob_total[bin_name]
    pct = err / total * 100
    print(f'{bin_name}: {err}/{total} ({pct:.1f}%)')

In [ ]:
# Interpretation for KR: Bayes Classifier Interpretation
print('\\n=== Interpretation for Course Work ===')
print('\\n1. Baseline PS (Pinnacle) performance:')
print(f'   - Log Loss: {test_baseline["log_loss"]:.4f} (target to beat)')
print(f'   - ROC AUC: {test_baseline["auc"]:.4f}')
print(f'   - Accuracy: {test_baseline["acc"]:.4f}')

print('\\n2. GaussianNB assumptions violations:')
print('   - Non-normal features: p_ps_imp, rank_diff have heavy tails')
print('   - Correlated features: p_ps_imp vs rank_points_diff (corr ~0.7)')
print('   - Scale differences: rank_diff range 3k vs p_ps_imp 0-1')
print('   - Conditional independence: violated by correlations')

print('\\n3. Model results:')
for name, metrics in models.items():
    delta_ll = metrics['log_loss'] - test_baseline['log_loss']
    print(f'   {name}: LL={metrics["log_loss"]:.4f} (Δ={delta_ll:+.4f}), AUC={metrics["auc"]:.4f}, ACC={metrics["acc"]:.4f}')
    if delta_ll < 0:
        print(f'      *** IMPROVED over baseline! ***')

print('\\n4. Weak spots of baseline (PS errors):')
print(f'   - Total errors: {len(baseline_errors)}/{len(test)} ({len(baseline_errors)/len(test)*100:.1f}%)')
print('   - Highest error rates: close matches and strong favorites (prob bins)')
print('   - By surface: higher on clay? (need to check data)')

print('\\n5. Conclusion:')
print('   - GNB struggles due to assumption violations (non-normal, correlated, scaled features)')
print('   - Calibration helps, but still not better than baseline')
print('   - CategoricalNB on surface alone is poor (no info)')
print('   - To improve: use proper feature engineering or other models (e.g., LR with regularization handles scale/corr)')